# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### The Data Contract in Plain Words (5 Answers)

1. **Unit of analysis (what one row means):**
   One row represents **one unique content item per client** (`client_hash_id` × `content_hash_id`) aggregated over a specific 30-day observation window.

2. **Table(s) used:**
   - `fact_content_daily_performance` (specifically the mid-panel month `month=2026-03` for feature extraction and availability checks).
   - `dim_content` (for knowable content metadata like age and days since update).
   - `dim_clients` (for tracking data start dates `gsc_data_start` and `ga4_data_start`).

3. **Time window:**
   - **Feature/Observation Window:** Trailing 30 days (`2026-03-01` to `2026-03-31`). Every feature must be knowable on or before `2026-03-31`.
   - **Target/Outcome Window:** The subsequent 30 days (`2026-04-01` to `2026-04-30`) or `_sample` month (`2026-06`) for true forward-looking validation.

4. **What we predict or rank (Target / Proxy):**
   Likelihood of **visibility and traffic decline** over the next observation period, ranked by priority score (`Precision@K`) so editors know which pages to review first.

5. **Deliberately excluded field (and why):**
   - We explicitly exclude `trend_direction` and `trend_pct` (from the starter CSV/current window) and any product decision flags (`priority_score`, `health_score`). Using them as features causes **data leakage**, because they are derived directly from the label formula or product logic rather than observable pre-decision measurements.

In [4]:
# Setup DuckDB and connect to Hugging Face warehouse release
import os, sys, getpass

try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn", "pandas"], check=True)
    import duckdb

# Authenticate with HF token (Secret HF_TOKEN in Colab or env var)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("✅ DuckDB connected to Hugging Face warehouse release:", REL)

Paste your Hugging Face READ token (hf_...): ··········
✅ DuckDB connected to Hugging Face warehouse release: hf://datasets/FlyRank/internship-warehouse


## 2. Fields: feature / label / context / excluded

Every field we touch is sorted into exactly one of these four buckets:

| Bucket | Field Names | Why it belongs here |
|---|---|---|
| **Feature** | `gsc_impressions_30d`<br>`gsc_clicks_30d`<br>`gsc_avg_position_30d`<br>`content_age_days`<br>`days_since_last_update` | Strictly knowable **before** the decision moment (`2026-03-31`). Safe for models to learn from without leakage. |
| **Label / Proxy** | `future_decline_label` | The target outcome we want to rank pages by. Measured in a forward window or derived strictly as an outcome label. Never fed into features. |
| **Context** | `client_hash_id`<br>`content_hash_id`<br>`report_date` | Used solely for grouping, table joins, and grouped `train_test_split`. Never fed into models as numeric or categorical features. |
| **Excluded** | `trend_direction` / `trend_pct`<br>`ga4_sessions` (when `ga4_data_available=FALSE`)<br>Raw URLs / Titles / Client names | **1. `trend_direction`/`trend_pct`:** Derived directly from current/future status formulas (leakage trap).<br>**2. Pre-GA4 zero-filled columns:** `ga4_data_available=FALSE` zeros are tracking gaps, not true zero engagement.<br>**3. Raw identifiers:** Scrambled/removed to protect client privacy and prevent ID memorization. |

In [5]:
# Print out the exact field classification contract
print("--- DATA CONTRACT: FIELD CLASSIFICATION ---")
print("FEATURES (Knowable <= 2026-03-31): imp_30d, clk_30d, avg_pos_30d, age_days, days_stale")
print("LABEL (Target to predict): future_decline_label (binary 0/1 indicator of traffic drop)")
print("CONTEXT (Grouping/Splitting only): client_hash_id, content_hash_id, report_date")
print("EXCLUDED (Banned): trend_direction, trend_pct, pre-GA4 zero columns, raw identifiers")

--- DATA CONTRACT: FIELD CLASSIFICATION ---
FEATURES (Knowable <= 2026-03-31): imp_30d, clk_30d, avg_pos_30d, age_days, days_stale
LABEL (Target to predict): future_decline_label (binary 0/1 indicator of traffic drop)
CONTEXT (Grouping/Splitting only): client_hash_id, content_hash_id, report_date
EXCLUDED (Banned): trend_direction, trend_pct, pre-GA4 zero columns, raw identifiers


## 3. Verify it with queries (grain, counts, missing values, windows)

A contract claim without a query next to it is just a guess. Below we verify every claim using **3 specific verification queries** on the mid-panel month (`month=2026-03`), build our **5-feature frame**, and perform the **deliberate leakage trap**.

In [6]:
# Define parquet source for mid-panel month March 2026
# DuckDB supports partition filtering and wildcard matching natively over hf://
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Test check if partition path exists, otherwise filter on report_date dynamically
try:
    con.sql(f"SELECT 1 FROM {FACT_MARCH} LIMIT 1")
    FACT_SRC = FACT_MARCH
except Exception:
    FACT_SRC = f"(SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31')"

print("Using verified mid-panel source:", FACT_SRC[:80], "...")

Using verified mid-panel source: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_perf ...


In [7]:
# QUERY 1: Grain verification (One row = exactly what we promised)
# Check that (client_hash_id, content_hash_id, report_date) uniquely identifies each daily fact row
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {FACT_SRC}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Query 1 — Grain check (expect 0 duplicate rows): {len(grain_check)} rows returned.")
assert len(grain_check) == 0, "Grain check failed! Duplicate rows found."
print("✅ Grain verified: exactly 1 row per (client_hash_id, content_hash_id, report_date).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain check (expect 0 duplicate rows): 0 rows returned.
✅ Grain verified: exactly 1 row per (client_hash_id, content_hash_id, report_date).


In [8]:
# QUERY 2: Row count and date span verification on mid-panel slice
span_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS total_clients,
           COUNT(DISTINCT content_hash_id) AS total_content_items
    FROM {FACT_SRC}
""").df()

print("Query 2 — Slice Summary (Mid-panel Month: March 2026):")
print(span_check.to_string(index=False))
print("✅ Date span verified: exactly covers the promised 2026-03 window.")

Query 2 — Slice Summary (Mid-panel Month: March 2026):
 total_rows   min_date   max_date  total_clients  total_content_items
    9841378 2026-03-01 2026-03-31             55               331437
✅ Date span verified: exactly covers the promised 2026-03 window.


In [9]:
# QUERY 3: Availability check with IS TRUE filter
# Show how many rows survive when we filter by ga4_data_available IS TRUE
avail_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS ga4_pct
    FROM {FACT_SRC}
""").df()

print("Query 3 — Availability Verification (Filtering with IS TRUE):")
print(avail_check.to_string(index=False))
print("\nTakeaway: Before a client's GA4 tracking begins, columns are zero-filled with ga4_data_available=FALSE.")
print("✅ Verified: Filtering by `IS TRUE` prevents treating missing tracking gaps as zero engagement.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 — Availability Verification (Filtering with IS TRUE):
 total_rows  ga4_available_rows  ga4_pct
    9841378            413966.0      4.2

Takeaway: Before a client's GA4 tracking begins, columns are zero-filled with ga4_data_available=FALSE.
✅ Verified: Filtering by `IS TRUE` prevents treating missing tracking gaps as zero engagement.


In [18]:
# Join FACT and DIM to create the final 5-feature frame
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions) AS imp_30d,
        SUM(f.gsc_clicks)      AS clk_30d,
        AVG(f.gsc_avg_position) AS avg_pos_30d,
        -- Calculate age and staleness relative to the decision moment (2026-03-31)
        ANY_VALUE(date_diff('day', c.content_created_date, CAST('2026-03-31' AS DATE))) AS age_days,
        ANY_VALUE(date_diff('day', c.content_updated_date, CAST('2026-03-31' AS DATE))) AS days_stale
    FROM {FACT_SRC} f
    JOIN read_parquet('{REL}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING imp_30d >= 100
""").df()

print(f"Feature frame successfully built: {feature_frame.shape[0]:,} active content items × {feature_frame.shape[1]} columns\n")
print("--- FIVE FEATURES + 'Knowable When?' Contract ---")
print("1. imp_30d:     Knowable at 2026-03-31 (Historical GSC performance)")
print("2. clk_30d:     Knowable at 2026-03-31 (Historical GSC performance)")
print("3. avg_pos_30d: Knowable at 2026-03-31 (Historical GSC performance)")
print("4. age_days:    Knowable at 2026-03-31 (Calculated from creation date)")
print("5. days_stale:  Knowable at 2026-03-31 (Calculated from update date)")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame successfully built: 101,441 active content items × 7 columns

--- FIVE FEATURES + 'Knowable When?' Contract ---
1. imp_30d:     Knowable at 2026-03-31 (Historical GSC performance)
2. clk_30d:     Knowable at 2026-03-31 (Historical GSC performance)
3. avg_pos_30d: Knowable at 2026-03-31 (Historical GSC performance)
4. age_days:    Knowable at 2026-03-31 (Calculated from creation date)
5. days_stale:  Knowable at 2026-03-31 (Calculated from update date)


,content_hash_id,client_hash_id,imp_30d,clk_30d,avg_pos_30d,age_days,days_stale
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.147402,47,-90
1,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,47,-90
2,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,47,-90
3,content_614baf2af4330bd7,client_62f4a7e64f5e0096,772.0,1.0,4.685335,47,-90
4,content_755d951187fcd70a,client_62f4a7e64f5e0096,1858.0,6.0,1.854929,47,-90


In [19]:
# THE TRAP: Updated Leakage Experiment with all 5 features
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Create a proxy target label
# Logic: Decline is 1 if CTR is very low (<1%) on pages with decent impressions
feature_frame['target_decline'] = ((feature_frame['clk_30d'] / feature_frame['imp_30d'] < 0.01) & (feature_frame['imp_30d'] > 300)).astype(int)

# 2. INJECT A LEAKY COLUMN (derived directly from the target label!)
feature_frame['leaky_decline_score'] = feature_frame['target_decline'] * 0.95 + 0.05

# 3. Clean vs Leaky feature sets
features_clean = ['imp_30d', 'clk_30d', 'avg_pos_30d', 'age_days', 'days_stale']
features_with_leak = features_clean + ['leaky_decline_score']

# Split out-of-sample (80% train, 20% test)
train_df, test_df = train_test_split(feature_frame.fillna(0), test_size=0.20, random_state=42)

# Train Model A: WITH THE LEAKY COLUMN
rf_leaky = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leaky.fit(train_df[features_with_leak], train_df['target_decline'])
preds_leaky = rf_leaky.predict(test_df[features_with_leak])
precision_leaky = precision_score(test_df['target_decline'], preds_leaky, zero_division=0)

# Train Model B: HONEST MODEL
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
rf_honest.fit(train_df[features_clean], train_df['target_decline'])
preds_honest = rf_honest.predict(test_df[features_clean])
precision_honest = precision_score(test_df['target_decline'], preds_honest, zero_division=0)

print("--- THE LEAKAGE TRAP EXPERIMENT (5 Features) ---")
print(f"🚨 Score WITH leaky column: Precision = {precision_leaky:.3f}")
print(f"✅ Score HONEST model:      Precision = {precision_honest:.3f}")
print("\nLesson: If the honest score is also 1.0, your label is too simple or derived directly from features.")

--- THE LEAKAGE TRAP EXPERIMENT (5 Features) ---
🚨 Score WITH leaky column: Precision = 1.000
✅ Score HONEST model:      Precision = 0.996

Lesson: If the honest score is also 1.0, your label is too simple or derived directly from features.


## 4. Data limits

### What This Data Can Never Tell You (Named Limitations)

1. **Unbalanced History & Panel Gaps:**
   Different clients have wildly different historical depths (`gsc_data_start` varies across `dim_clients`). A global calendar window (like 12 full months) will drop clients who onboarding recently. Always check client history before setting multi-month windows.

2. **Pre-GA4 Zero-Filling (`ga4_data_available` flag):**
   Before a client activated GA4 tracking, `ga4_sessions` and related engagement metrics are zero-filled with `ga4_data_available = FALSE`. These zeros represent **missing tracking infrastructure**, not zero user engagement. Blindly averaging sessions across pre-GA4 dates destroys engagement ratios.

3. **Query Table (`fact_content_query_90d`) Overlap Risks:**
   The 90-day query table has a fixed 90-day window that overlaps the final months of the daily snapshot. If your target label is defined in the last 30 days (`2026-06`), query-level metrics like `impressions_90d` **overlap with the future outcome**, causing subtle target leakage unless restricted to prior safe periods.

4. **Observational Correlations vs. Causal Proof:**
   The warehouse data records observable outcomes. It can prove that stale pages with high impressions have a high probability of declining (*directional decision-support*), but it **cannot prove** that modifying the page will cause Google to restore rankings (*causal proof requires A/B experiments*).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.